Offline teaching language view. Code, existing outputs and configuration retain the accepted reader identity. Fixed filing facts and reference figures are historical teaching inputs; new constructed calculations are not market-strategy reproduction.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch10_markets_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 10: market instruments and cash flows

All inputs are constructed. Currency, quote direction, contract multiplier and event times are stated below. Functions are shared with `code/ch10_markets.py`; start inside the reader tree and run in order so the bootstrap sets the package root.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "code" / "ch10_markets.py").exists():
    raise FileNotFoundError("请从 quant_course 项目根目录启动 Notebook")
sys.path.insert(0, str(PROJECT_ROOT / "code"))
from quant_course.market_instruments import (
    bond_coupon_cashflow,
    covered_fx_forward_quote,
    equity_total_return,
    fx_convert_base_to_quote,
    futures_variation_margin_usd,
    interest_rate_swap_net_cashflow,
    listed_option_pnl,
    repo_cash_advanced,
)
from ch10_markets import build_cashflow_timeline

## Equities and bonds: entitlement is distinct from payment

The equity example buys 100 shares at USD 50 per share, ends at USD 50.20 and includes a dividend of USD 0.50 per share. The function gives a 1.4% simple holding-period total return without modeling the ex-date, taxes or payment timing. The bond function gives each coupon payment: USD 1,000 face value, a 6% annual rate and two payments per year give USD 30 per payment. The payer-fixed swap example gives −USD 1,000 for its quarter.

In [ ]:
equity_return = equity_total_return(100, 50.00, 50.20, 0.50)
semiannual_coupon = bond_coupon_cashflow(1_000.0, 0.06, 2)
swap_cashflow = interest_rate_swap_net_cashflow(1_000_000.0, 0.04, 0.036, 0.25)
assert abs(equity_return - 0.014) < 1e-12
assert abs(semiannual_coupon - 30.0) < 1e-12
assert abs(swap_cashflow - (-1_000.0)) < 1e-12
print(f"股票简单总收益：{equity_return:.2%}")
print(f"半年票息：USD {semiannual_coupon:.2f}")
print(f"互换首期净现金流：USD {swap_cashflow:.2f}")

## FX: read the quote direction first

Here `USD/EUR=1.10` means USD 1.10 per EUR 1. EUR 10,000 converts to USD 11,000. The forward reference assumes equal maturities, simple interest and no frictions: 1.10×1.025/1.015 for the half-year. It is not an executable dealer quote.

In [ ]:
spot_usd_per_eur = 1.10
converted_usd = fx_convert_base_to_quote(10_000.0, spot_usd_per_eur)
forward_usd_per_eur = covered_fx_forward_quote(spot_usd_per_eur, 0.05, 0.03, 0.5)
assert converted_usd == 11_000.0
assert abs(forward_usd_per_eur - (1.10 * 1.025 / 1.015)) < 1e-12
print(f"EUR 10,000 -> USD {converted_usd:,.2f} at the stated spot")
print(f"simple parity reference: USD per EUR {forward_usd_per_eur:.6f}")

## MES, options and repo: convert contract units to cash

Futures settlement changes are index points multiplied by USD per point, giving USD 100, −200 and 150. The option uses matching USD-per-share quotes and an explicit 100-share multiplier; expiry payoff less premium gives USD 600. Repo haircut arithmetic gives USD 950,000 cash against USD 1 million of collateral; it does not model a complete financing contract.

In [ ]:
mes_vm = futures_variation_margin_usd(1, 5.0, [20.0, -40.0, 30.0])
call_pnl = listed_option_pnl(110.0, 100.0, 4.0, 1, 100, "call")
repo_advance = repo_cash_advanced(1_000_000.0, 0.05)
assert mes_vm == (100.0, -200.0, 150.0)
assert call_pnl == 600.0
assert repo_advance == 950_000.0
print(f"MES variation cash flows: {mes_vm}; net USD {sum(mes_vm):.2f}")
print(f"long call expiry P&L: USD {call_pnl:.2f}")
print(f"repo cash advanced: USD {repo_advance:,.2f}")

## Event diagram

The new timeline is written under `RUN_DIR/figures/a3`. Horizontal position gives event order rather than calendar distance; lanes classify instruments rather than amounts. Labels use constructed USD amounts. The blue diamond denotes dividend entitlement, not cash receipt that day. Chinese print labels name the equity total return, semiannual coupon and swap cash flow computed above.

In [ ]:
figure_path = build_cashflow_timeline(RUN_DIR / "figures/a3/ch10_cashflow_timeline.png")
print(f"wrote {figure_path.relative_to(PROJECT_ROOT)}")
from ch10_markets import build_cashflow_timeline